# Module 03 — AI Agents
## Lesson 10 — Build a Real Agent from Scratch in Python

This lesson combines the mechanisms from the previous lessons into one coherent implementation before we introduce an orchestration framework.

**Architecture principle:** the model owns bounded decisions; the host owns tools, state, budgets, retries, validation, approvals, and execution authority.


## Scenario

We will build a small travel assistant for a goal such as:

> Plan a family-friendly afternoon in Melbourne tomorrow. Check the weather and daylight. Prefer outdoor activities if conditions are suitable. If you recommend something that needs a reservation, ask before reserving it for three people.

The agent can call live weather/sun tools, inspect a small activity catalogue, and request a **simulated** reservation. The reservation is never executed without host-controlled approval.


## What is now integrated?

| Earlier lesson | In this implementation |
|---|---|
| Tool calling | explicit schemas + allowlisted registry |
| Agent loop | repeated model decision → action → observation |
| State & memory | `AgentState` + selected memory context |
| Planning | optional concise external plan |
| Retries | bounded transient HTTP retries |
| Budgets | step, tool-call, repeated-action limits |
| Human-in-the-loop | `WAITING_FOR_APPROVAL` before reservation |
| Workflow vs agent | deterministic shell around agentic decisions |


## Keep reusable orchestration outside the notebook

The implementation lives in `agent.py` so we can test deterministic host logic normally. The notebook remains the teaching surface rather than becoming a giant untestable code cell.


In [ ]:
from agent import (
    RunBudget,
    RunStatus,
    run_agent,
    resume_after_approval,
)


## Architecture

```text
selected memory + goal
        ↓
optional external plan
        ↓
┌──────── host-controlled agent loop ────────┐
│ check budgets                               │
│        ↓                                    │
│ model decides next action                   │
│        ↓                                    │
│ validate tool + args + repeated action      │
│        ↓                                    │
│ read-only? ─ yes → execute/retry → observe  │
│        │                                     │
│        no                                    │
│        ↓                                     │
│ side effect → WAITING_FOR_APPROVAL           │
└──────────────────────────────────────────────┘
```

The model never executes Python directly and cannot extend its own budgets or bypass approval.


## Inspect the host-owned objects

`agent.py` defines explicit `Tool`, `AgentState`, `RunBudget`, `RunUsage`, `ActionProposal`, and `AgentResult` types. These are application state, not prompt prose.

The tool registry distinguishes read-only tools from the simulated side-effecting `reserve_activity` tool.


In [ ]:
from agent import TOOLS

for name, tool in TOOLS.items():
    print(f"{name:20} side_effect={tool.side_effect}")


## Memory is selected context, not replayed history

For this run we deliberately inject only stable preferences that are relevant to travel. We do not replay old weather results or unrelated conversations.


In [ ]:
memory = {
    "travel_style": "family-friendly",
    "outdoor_preference": "prefer outdoors when weather is suitable",
}


## Use explicit bounded policy

These numbers are intentionally small so the agent cannot continue indefinitely. Retry attempts are separate from model/tool budgets.


In [ ]:
budget = RunBudget(
    max_steps=6,
    max_tool_calls=6,
    max_same_action=2,
    retry_attempts=3,
)


## Run the integrated agent

Set `use_planning=True` for this broader goal. Planning is optional orchestration; for a simple one-tool question you should leave it off.


In [ ]:
goal = (
    "Plan a family-friendly afternoon in Melbourne tomorrow. "
    "Check the weather and daylight. Prefer outdoor activities if conditions are suitable. "
    "If you recommend something that needs a reservation, ask before reserving it for three people."
)

result = run_agent(
    goal,
    budget=budget,
    memory_context=memory,
    use_planning=True,
)

print("status:", result.status.value)
print("reason:", result.stop_reason.value)
print("steps:", result.state.usage.steps)
print("tool calls:", result.state.usage.tool_calls)
print("plan:", result.state.plan)
print("observations:", len(result.state.observations))
if result.answer:
    print("\nANSWER\n", result.answer)
if result.approval:
    print("\nAPPROVAL REQUIRED\n", result.approval)


## Two legitimate outcomes

The model may recommend a no-reservation activity and finish normally. Or it may request `reserve_activity`, in which case the host returns `WAITING_FOR_APPROVAL` instead of executing the side effect.

That variation is expected: the model owns a bounded decision, while the application owns the safety boundary.


## Resume an approved or denied side effect

Only run the next cell if `result.status == RunStatus.WAITING_FOR_APPROVAL`. Change `approved` to `False` to feed a denial back to the agent instead.


In [ ]:
if result.status == RunStatus.WAITING_FOR_APPROVAL:
    resumed = resume_after_approval(
        result,
        approved=True,  # try False as well
        budget=budget,
    )
    print("status:", resumed.status.value)
    print("reason:", resumed.stop_reason.value)
    if resumed.answer:
        print("\nANSWER\n", resumed.answer)
else:
    print("No approval checkpoint was needed in this run.")


## Inspect the trace, not only the answer

The application keeps structured observations separately from provider protocol history. This is what makes the run debuggable and testable.


In [ ]:
for i, item in enumerate(result.state.observations, start=1):
    print(f"\nObservation {i}")
    print("tool:", item["tool"])
    print("arguments:", item["arguments"])
    print("result:", item["observation"])


## What is deliberately still simple?

This is a teaching implementation, not a complete agent platform. A production version would normally add durable checkpoint storage, richer provider usage/cost accounting, cancellation/deadlines, audit records, approval identity/expiry, observability, concurrent-run isolation, stronger typed tool validation, and systematic evaluation.

Those gaps are useful: when we compare frameworks next, we can ask which mechanics they actually remove and which responsibilities still belong to our application.


## Exercises

1. Ask only for tomorrow's Melbourne weather with planning disabled. Confirm the agent needs less orchestration.
2. Set `max_tool_calls=1` and use the broader goal. Inspect the structured stop result.
3. Deny a reservation and observe whether the model can adapt without executing the side effect.
4. Add a new read-only `get_activity_details` tool without changing the main loop.
5. Add a memory preference such as `avoid_boats=true` and decide whether it belongs in durable memory.
6. Replace the travel tools with engineering tools such as service health, logs, and deployment history. Keep the same host architecture.

Next: **Lesson 11 — Compare Agent Frameworks**.
